In [1]:
from pathlib import Path
from typing import List, Tuple, Dict
import json, random, itertools, os, gc, statistics as st
import seqeval

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, concatenate_datasets
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans

import torch
from transformers import (
    AutoTokenizer,
        AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

from seqeval.metrics import f1_score, classification_report
from evaluate import load as load_metric
from tqdm.auto import tqdm

c:\Users\user\miniconda3\envs\lora-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configuração e Verificação Inicial

In [2]:
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

MODEL_NAME = "dbmdz/bert-base-cased-finetuned-conll03-english"

In [3]:
def read_conll(path):
    tokens, tags = [], []
    sent_tokens, sent_tags = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip()
            if not line:
                if tokens:
                    sent_tokens.append(tokens)
                    sent_tags.append(tags)
                    tokens, tags = [], []
            else:
                tok, _, _, ner = line.split()
                tokens.append(tok)
                tags.append(ner)
    # último sentença
    if tokens:
        sent_tokens.append(tokens)
        sent_tags.append(tags)
    return sent_tokens, sent_tags

In [4]:
def read_conll(path: Path, start_sentence_id: int = 0):
    """
    Lê arquivos CoNLL/ CleanCoNLL:
      • usa parts[0] como token
      • usa parts[-1] como rótulo NER (corrigido)
      • ignora linhas '-DOCSTART- …'
    """
    tokens, tags, sent_ids = [], [], []
    cur_toks, cur_tags = [], []
    sid = start_sentence_id

    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.rstrip()

            if not line:  # fim da sentença
                if cur_toks:
                    tokens.append(cur_toks)
                    tags.append(cur_tags)
                    sent_ids.append(sid)
                    sid += 1
                    cur_toks, cur_tags = [], []
                continue

            parts = line.split()
            if parts[0] == "-DOCSTART-":  # pula marcador de doc
                continue

            tok, ner = parts[0], parts[-1]  # 1ª e última coluna
            cur_toks.append(tok)
            cur_tags.append(ner)

    if cur_toks:  # última sentença
        tokens.append(cur_toks)
        tags.append(cur_tags)
        sent_ids.append(sid)

    return tokens, tags, sent_ids, sid  # devolve sid para continuar contagem

In [5]:
def load_cleanconll(base_dir=None, keep_sentence_id=True):
    # 1. Resolva o diretório da forma mais robusta possível
    if base_dir is None:
        base_dir = Path.home() / "Documents" / "mestrado" / "ner_splits" / "data"
    else:
        base_dir = Path(base_dir).expanduser()

    FILES = {
        "train": "cleanconll.train",
        "dev"  : "cleanconll.dev",
        "test" : "cleanconll.test",
    }

    # 2. Verifique se todos os arquivos existem antes de começar
    missing = [fname for fname in FILES.values() if not (base_dir / fname).exists()]
    if missing:
        raise FileNotFoundError(f"Arquivos não encontrados em {base_dir}: {', '.join(missing)}")

    splits, sid = {}, 0
    for split, fname in FILES.items():
        toks, labs, sids, sid = read_conll(base_dir / fname, sid)
        data = {"tokens": toks, "ner_tags": labs}
        if keep_sentence_id:
            data["sentence_id"] = sids
        splits[split] = Dataset.from_dict(data)

    return DatasetDict(splits)

In [6]:
cleanconll_ds = load_cleanconll()  # pronto!
print(cleanconll_ds)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'sentence_id'],
        num_rows: 13957
    })
    dev: Dataset({
        features: ['tokens', 'ner_tags', 'sentence_id'],
        num_rows: 3233
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'sentence_id'],
        num_rows: 3427
    })
})


In [7]:
cleanconll_ds["train"] = cleanconll_ds["train"].add_column(
    "split", ["train"] * len(cleanconll_ds["train"])
)
cleanconll_ds["dev"] = cleanconll_ds["dev"].add_column(
    "split", ["dev"] * len(cleanconll_ds["dev"])
)
cleanconll_ds["test"] = cleanconll_ds["test"].add_column(
    "split", ["test"] * len(cleanconll_ds["test"])
)

cleanconll_full = concatenate_datasets(
    [
        cleanconll_ds["train"],
        cleanconll_ds["dev"],
        cleanconll_ds["test"],
    ]
)

In [8]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in cleanconll_full["ner_tags"] for l in sent})
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [9]:
id2label

{0: 'B-LOC',
 1: 'B-MISC',
 2: 'B-ORG',
 3: 'B-PER',
 4: 'I-LOC',
 5: 'I-MISC',
 6: 'I-ORG',
 7: 'I-PER',
 8: 'O'}

In [10]:
NUM_LABELS

9

# Splits

In [11]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [12]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [13]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [14]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all   = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(idx_all,
                                           test_size=pct_test + pct_val,
                                           stratify=(sent_lens//5),  # bin len
                                           random_state=seed)
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test*len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val*len(dataset))
        val_idx  = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test*len(dataset))]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(dataset: Dataset,
                    pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w,c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks)
                        for toks in dataset["tokens"]])
    rare_idx   = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test*len(dataset))
    n_val  = int(pct_val *len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test),
                          replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx  = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(dataset: Dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = (np.array([len(t) for t in dataset["tokens"]]) // 5)
    train_idx, temp_idx = train_test_split(idx, test_size=pct_test+pct_val,
                                          stratify=strat, random_state=seed)
    val_rel = pct_val / (pct_test+pct_val)
    val_idx, test_idx = train_test_split(temp_idx, test_size=1-val_rel,
                                         stratify=strat[temp_idx],
                                         random_state=seed)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(toks) for toks in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    print(f"Selecionando {int(pct_test*len(dataset))} sentenças para teste...")
    while len(test_idx) < int(pct_test*len(dataset)):
        print(f"  {len(test_idx)} selecionadas...")
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test*len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val*len(dataset))
    val_idx  = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [15]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42,
                   bin_size: int = 10) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all   = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = (sent_lens // bin_size)
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:                       # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired  = int(pct_val  * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(resto_idx,
                           size=n_test_desired - len(test_idx),
                           replace=False)
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx  = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [16]:
def std_split(dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42,
              bin_size: int = 5) -> DatasetDict:
    """
    Split 80-10-10 robusto.
      • Tenta estratificar por comprimento // bin_size.
      • Se houver classes com <2 amostras, recua p/ split aleatório.
    """
    idx  = np.arange(len(dataset))
    bins = (np.array([len(t) for t in dataset["tokens"]]) // bin_size)

    try:
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            stratify=bins,
            random_state=seed,
        )
    except ValueError:                       # classes muito pequenas
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # fraciona temp em val / test mantendo proporção desejada
    val_share = pct_val / (pct_test + pct_val)
    try:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            stratify=bins[temp_idx],
            random_state=seed,
        )
    except ValueError:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [17]:
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Versão robusta: nunca “trava” antes de atingir n_test.
    Seleciona blocos de k sentenças mais distantes do centro iterativamente.
    """
    # -------------------------------- embeds -------------------------------
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(t) for t in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    tree = BallTree(emb, leaf_size=40)

    n_test = int(pct_test * len(dataset))
    n_val  = int(pct_val  * len(dataset))

    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)

    print(f"Selecionando {n_test} sentenças para teste…")
    step = 0
    while len(test_idx) < n_test and idx_pool:
        if step % 5 == 0:
            print(f"  {len(test_idx)} selecionadas…")
        step += 1

        pool_list = list(idx_pool)
        center = emb[pool_list].mean(0, keepdims=True)

        # distância euclidiana ao centro global restante
        dists, _ = tree.query(center, k=len(pool_list))
        farthest_local_idx = int(dists.argmax())
        farthest_global_idx = pool_list[farthest_local_idx]

        # número efetivo de vizinhos
        k_eff = min(k, len(idx_pool))
        nn = set(tree.query([emb[farthest_global_idx]],
                            k=k_eff,
                            return_distance=False)[0])

        # adiciona vizinhos ainda não selecionados
        for j in nn:
            if j in idx_pool and len(test_idx) < n_test:
                test_idx.append(j)
                idx_pool.remove(j)

        # Se nada foi adicionado (pode acontecer quando sobram <k únicos)
        if farthest_global_idx not in test_idx:
            test_idx.append(farthest_global_idx)
            idx_pool.remove(farthest_global_idx)

    # ------------------------ validação e treino ---------------------------
    val_idx = rng.choice(list(idx_pool), size=n_val, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [18]:
def standard_split_conll(dataset):
    return cleanconll_ds

In [19]:
# standard_split = standard_split_conll(cleanconll_full)
# print('std')
# # random_splt = random_splits(cleanconll_full)
# # print('random')
# heur_len = heur_len_split(cleanconll_full)
# print("heur_len")
# heur_rare = heur_rare_split(cleanconll_full)
# print("heur_rare")
# advers = adversarial_split(cleanconll_full)
# print("advs")
# loc = loc_split(cleanconll_full)
# print("loc")
# semantic = semantic_cluster_split(cleanconll_full)
# print("semantic")
# reverse = reverse_curriculum_split(cleanconll_full)
# print("reverse")

In [20]:
gc.collect()  # força o GC do Python
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

# Experimentos

In [21]:
from sklearn.metrics import f1_score as skl_f1

In [22]:
def train_ner_with_split(
    dataset: Dataset,
    split: str,  # "loc" | "semantic" | "reverse" | func
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    model_ckpt: str = "Davlan/distilbert-base-multilingual-cased-ner-hrl",
    training_args_kwargs: dict | None = None,
    split_kwargs: dict | None = None,
    seed: int = 42,
):
    """
    Treina um modelo de NER usando a estratégia de split desejada.
    Retorna (trainer, métricas_test).
    """
    # 1. Escolhe função de split ------------------------------------------------
    if callable(split):
        split_fn = split
    else:
        _map = {
            "loc": loc_split,
            "reverse": reverse_curriculum_split,
            "semantic": semantic_cluster_split,
            "heur_len": heur_len_split,
            "heur_rare": heur_rare_split,
            "std": standard_split_conll,
            "advs": adversarial_split,
        }
        if split not in _map:
            raise ValueError(f"split='{split}' não reconhecido.")
        split_fn = _map[split]

    split_kwargs = split_kwargs or {}
    ds = split_fn(
        dataset, pct_test=pct_test, pct_val=pct_val, seed=seed, **split_kwargs
    )  # train/val/test

    # 2. Tokenizer e modelo -----------------------------------------------------
    label_list = sorted({l for labels in dataset["ner_tags"] for l in labels})
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for l, i in label2id.items()}

    num_labels = len(label_list)
    tok = AutoTokenizer.from_pretrained(model_ckpt)
    model = AutoModelForTokenClassification.from_pretrained(
        model_ckpt,
        num_labels=num_labels,  # ← adapta o tamanho
        ignore_mismatched_sizes=True,  # ← descarta pesos velhos da head
    )
    model.config.id2label = id2label
    model.config.label2id = label2id

    # 3. Mapeamento label↔id ----------------------------------------------------
    # label_list = sorted(
    #     {l for labels in dataset["ner_tags"] for l in labels if l != "O"}
    # )

    # 4. Tokenização + alinhamento ---------------------------------------------
    # def tok_function(ex):
    #     return tok(
    #         ex["tokens"], is_split_into_words=True, truncation=True, padding=False
    #     )

    # def align_labels(ex):
    #     word_ids = ex.word_ids()
    #     labels = []
    #     for w in word_ids:
    #         if w is None:
    #             labels.append(-100)
    #         else:
    #             labels.append(label2id.get(ex["ner_tags"][w], 0))
    #     ex["labels"] = labels
    #     return ex

    # ds_tok = ds.map(tok_function, batched=True)
    # ds_tok = ds_tok.map(align_labels)

    def tokenize_and_align_labels(examples, label_all_tokens=False):
        tokenized = tok(examples["tokens"], is_split_into_words=True, truncation=True)

        labels_batch = []
        for i, word_labels in enumerate(examples["ner_tags"]):
            word_ids = tokenized.word_ids(batch_index=i)  # <- aqui sim
            label_ids = []
            previous_word_idx = None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)  # máscara
                elif word_idx != previous_word_idx:
                    label_ids.append(label2id[word_labels[word_idx]])
                else:
                    # marca sub-tokens; mude para `label2id[...]`
                    # se quiser repetir label em todos os sub-tokens
                    label_ids.append(
                        label2id[word_labels[word_idx]] if label_all_tokens else -100
                    )
                previous_word_idx = word_idx
            labels_batch.append(label_ids)

        tokenized["labels"] = labels_batch
        return tokenized

    ds_tok = ds.map(
        tokenize_and_align_labels, batched=True, remove_columns=ds["train"].column_names
    )
    # 5. Métrica (seqeval) ------------------------------------------------------
    seqeval = load_metric("seqeval")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        sent_preds, sent_labels = [], []  # p/ seqeval
        flat_preds, flat_labels = [], []  # p/ sklearn

        for p_row, l_row in zip(preds, labels):
            p_sent, l_sent = [], []
            for pi, li in zip(p_row, l_row):
                if li != -100:
                    lbl_true = id2label[li]
                    lbl_pred = id2label[pi]
                    p_sent.append(lbl_pred)
                    l_sent.append(lbl_true)
                    flat_preds.append(lbl_pred)
                    flat_labels.append(lbl_true)
            sent_preds.append(p_sent)
            sent_labels.append(l_sent)

        # métricas seqeval (micro F1 = overall_f1)
        seqeval_metrics = seqeval.compute(
            predictions=sent_preds,
            references=sent_labels,
        )

        # métricas sklearn
        f1_micro = skl_f1(flat_labels, flat_preds, average="micro", zero_division=0)
        f1_macro = skl_f1(flat_labels, flat_preds, average="macro", zero_division=0)
        f1_weighted = skl_f1(
            flat_labels, flat_preds, average="weighted", zero_division=0
        )

        return {
            **seqeval_metrics,  # overall_precision / recall / f1
            "f1_micro": f1_micro,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
        }

    # 6. Args de treinamento ----------------------------------------------------
    args_defaults = dict(
        # output_dir=f"ner-{split}",
        # estratégia de avaliação + salvamento
        eval_strategy="epoch",  # novo nome (4.52+)
        save_strategy="no",
        # load_best_model_at_end=True,
        metric_for_best_model="overall_f1",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        seed=seed,
        report_to="none",
    )

    if training_args_kwargs:
        args_defaults.update(training_args_kwargs)
    args = TrainingArguments(**args_defaults)

    data_collator = DataCollatorForTokenClassification(tokenizer=tok, padding=True)

    # 7. Trainer ---------------------------------------------------------------
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["val"],
        tokenizer=tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # 8. Avaliação final --------------------------------------------------------
    test_metrics = trainer.evaluate(eval_dataset=ds_tok["test"])
    return trainer, test_metrics

In [23]:
splits = ["loc", "reverse", "semantic", "heur_len", "heur_rare", "advs"]

In [24]:
results = {}
trainer_all = {}
s = splits[0]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: loc


C:\Users\user\AppData\Local\Temp\ipykernel_13964\4053031328.py:28: RuntimeWarning: invalid value encountered in divide
  topk[i] = (inter / uni).mean()
Map: 100%|██████████| 4123/4123 [00:00<00:00, 14422.31 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_13964\1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.165900,0.055178,"{'precision': 0.9415841584158415, 'recall': 0.9453280318091452, 'f1': 0.943452380952381, 'number': 1006}","{'precision': 0.8450704225352113, 'recall': 0.8318890814558059, 'f1': 0.8384279475982533, 'number': 577}","{'precision': 0.8728382502543235, 'recall': 0.896551724137931, 'f1': 0.8845360824742268, 'number': 957}","{'precision': 0.9495798319327731, 'recall': 0.9674657534246576, 'f1': 0.9584393553859203, 'number': 1168}",0.911490,0.922060,0.916745,0.984987,0.984987,0.913046,0.985028
2,0.015700,0.061567,"{'precision': 0.907563025210084, 'recall': 0.9662027833001988, 'f1': 0.9359653346172364, 'number': 1006}","{'precision': 0.8901303538175046, 'recall': 0.82842287694974, 'f1': 0.8581687612208259, 'number': 577}","{'precision': 0.9218061674008811, 'recall': 0.8746081504702194, 'f1': 0.8975871313672922, 'number': 957}","{'precision': 0.9634353741496599, 'recall': 0.9700342465753424, 'f1': 0.9667235494880547, 'number': 1168}",0.926327,0.922330,0.924324,0.986027,0.986027,0.913214,0.985898
3,0.006800,0.062758,"{'precision': 0.94140625, 'recall': 0.9582504970178927, 'f1': 0.9497536945812808, 'number': 1006}","{'precision': 0.8625, 'recall': 0.8370883882149047, 'f1': 0.8496042216358839, 'number': 577}","{'precision': 0.8953846153846153, 'recall': 0.9122257053291536, 'f1': 0.9037267080745341, 'number': 957}","{'precision': 0.9699570815450643, 'recall': 0.9674657534246576, 'f1': 0.9687098156879554, 'number': 1168}",0.926423,0.930421,0.928418,0.987197,0.987197,0.924877,0.987202
4,0.002900,0.068002,"{'precision': 0.9538152610441767, 'recall': 0.9443339960238568, 'f1': 0.949050949050949, 'number': 1006}","{'precision': 0.8784029038112523, 'recall': 0.8388214904679376, 'f1': 0.8581560283687943, 'number': 577}","{'precision': 0.882587064676617, 'recall': 0.9268547544409613, 'f1': 0.9041794087665648, 'number': 957}","{'precision': 0.9551986475063398, 'recall': 0.9674657534246576, 'f1': 0.9612930667800935, 'number': 1168}",0.923963,0.930690,0.927314,0.986872,0.986872,0.921696,0.986877
5,0.001200,0.066488,"{'precision': 0.9512437810945273, 'recall': 0.9502982107355865, 'f1': 0.9507707608155146, 'number': 1006}","{'precision': 0.8717047451669596, 'recall': 0.8596187175043327, 'f1': 0.8656195462478184, 'number': 577}","{'precision': 0.905894519131334, 'recall': 0.9153605015673981, 'f1': 0.9106029106029105, 'number': 957}","{'precision': 0.9626485568760611, 'recall': 0.9708904109589042, 'f1': 0.9667519181585678, 'number': 1168}",0.930895,0.933657,0.932274,0.987652,0.987652,0.926423,0.987644


F1 Macro: 0.9359031461911087
F1 Micro: 0.9912086597691525
F1 Weighted: 0.9911689937852941
{'eval_loss': 0.05156797915697098, 'eval_LOC': {'precision': 0.9692393736017897, 'recall': 0.9569298729983434, 'f1': 0.9630452903584329, 'number': 1811}, 'eval_MISC': {'precision': 0.879162702188392, 'recall': 0.8716981132075472, 'f1': 0.8754144954997632, 'number': 1060}, 'eval_ORG': {'precision': 0.913573407202216, 'recall': 0.9321650650084794, 'f1': 0.9227756015668719, 'number': 1769}, 'eval_PER': {'precision': 0.9811788013868251, 'recall': 0.9802078179119248, 'f1': 0.9806930693069307, 'number': 2021}, 'eval_overall_precision': 0.9435689629296112, 'eval_overall_recall': 0.9438522744332682, 'eval_overall_f1': 0.9437105974181927, 'eval_overall_accuracy': 0.9912086597691525, 'eval_f1_micro': 0.9912086597691525, 'eval_f1_macro': 0.9359031461911087, 'eval_f1_weighted': 0.9911689937852941, 'eval_runtime': 13.3976, 'eval_samples_per_second': 307.742, 'eval_steps_per_second': 19.257, 'epoch': 5.0}




20

In [25]:
!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

Acesso negado - .
Arquivo n�o encontrado  - -NAME
Arquivo n�o encontrado  - -EXEC
Arquivo n�o encontrado  - RM
Arquivo n�o encontrado  - -RF
Arquivo n�o encontrado  - {}
Arquivo n�o encontrado  - +


In [26]:
import time

In [27]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()


time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

Acesso negado - .
Arquivo n�o encontrado  - -NAME
Arquivo n�o encontrado  - -EXEC
Arquivo n�o encontrado  - RM
Arquivo n�o encontrado  - -RF
Arquivo n�o encontrado  - {}
Arquivo n�o encontrado  - +


In [28]:
results = {}
trainer_all = {}
s = splits[1]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: reverse


Map: 100%|██████████| 4123/4123 [00:00<00:00, 10658.11 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_13964\1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.151100,0.040802,"{'precision': 0.9793962625778629, 'recall': 0.9596244131455399, 'f1': 0.9694095328432536, 'number': 2130}","{'precision': 0.9014423076923077, 'recall': 0.7961783439490446, 'f1': 0.8455467869222097, 'number': 471}","{'precision': 0.8684807256235828, 'recall': 0.9433497536945813, 'f1': 0.9043683589138134, 'number': 812}","{'precision': 0.9872557349192863, 'recall': 0.977291841883936, 'f1': 0.9822485207100592, 'number': 1189}",0.952872,0.944589,0.948712,0.989340,0.989340,0.934295,0.989126
2,0.013300,0.047935,"{'precision': 0.9757751937984496, 'recall': 0.9455399061032864, 'f1': 0.9604196471149261, 'number': 2130}","{'precision': 0.9368686868686869, 'recall': 0.7876857749469215, 'f1': 0.8558246828143022, 'number': 471}","{'precision': 0.8511576626240352, 'recall': 0.9507389162561576, 'f1': 0.898196625945317, 'number': 812}","{'precision': 0.9679012345679012, 'recall': 0.9890664423885618, 'f1': 0.978369384359401, 'number': 1189}",0.945657,0.941547,0.943598,0.989399,0.989399,0.940828,0.989273
3,0.005000,0.091493,"{'precision': 0.9785564853556485, 'recall': 0.8784037558685446, 'f1': 0.925779317169718, 'number': 2130}","{'precision': 0.9004629629629629, 'recall': 0.8259023354564756, 'f1': 0.8615725359911407, 'number': 471}","{'precision': 0.7191216834400732, 'recall': 0.9679802955665024, 'f1': 0.8251968503937008, 'number': 812}","{'precision': 0.9808013355592654, 'recall': 0.9882253994953742, 'f1': 0.9844993715961456, 'number': 1189}",0.910680,0.917210,0.913933,0.985816,0.985816,0.932043,0.986097
4,0.001500,0.089659,"{'precision': 0.9768041237113402, 'recall': 0.8896713615023474, 'f1': 0.9312039312039312, 'number': 2130}","{'precision': 0.9144893111638955, 'recall': 0.8174097664543525, 'f1': 0.8632286995515696, 'number': 471}","{'precision': 0.7476007677543186, 'recall': 0.9593596059113301, 'f1': 0.8403451995685005, 'number': 812}","{'precision': 0.9800664451827242, 'recall': 0.992430613961312, 'f1': 0.9862097785206854, 'number': 1189}",0.920122,0.921121,0.920621,0.986616,0.986616,0.933862,0.986719
5,0.000800,0.087722,"{'precision': 0.9795605518650996, 'recall': 0.9, 'f1': 0.9380964032297529, 'number': 2130}","{'precision': 0.9216152019002375, 'recall': 0.8237791932059448, 'f1': 0.8699551569506727, 'number': 471}","{'precision': 0.7636186770428015, 'recall': 0.9667487684729064, 'f1': 0.8532608695652173, 'number': 812}","{'precision': 0.9825145711906744, 'recall': 0.992430613961312, 'f1': 0.9874476987447698, 'number': 1189}",0.926850,0.927857,0.927354,0.987800,0.987800,0.939446,0.987925


F1 Macro: 0.8983627923411217
F1 Micro: 0.9747944481745237
F1 Weighted: 0.9737913413599811
{'eval_loss': 0.1915094256401062, 'eval_LOC': {'precision': 0.8921148135928736, 'recall': 0.9156789705384355, 'f1': 0.9037433155080214, 'number': 2953}, 'eval_MISC': {'precision': 0.9234248788368336, 'recall': 0.7311332821693528, 'f1': 0.8161050828098229, 'number': 3909}, 'eval_ORG': {'precision': 0.7339743589743589, 'recall': 0.9020821609454136, 'f1': 0.8093915677859126, 'number': 1777}, 'eval_PER': {'precision': 0.9563782991202346, 'recall': 0.9860166288737717, 'f1': 0.9709713435057685, 'number': 2646}, 'eval_overall_precision': 0.8854865011777496, 'eval_overall_recall': 0.8661054497120071, 'eval_overall_f1': 0.8756887515118936, 'eval_overall_accuracy': 0.9747944481745237, 'eval_f1_micro': 0.9747944481745237, 'eval_f1_macro': 0.8983627923411217, 'eval_f1_weighted': 0.9737913413599811, 'eval_runtime': 14.2724, 'eval_samples_per_second': 288.88, 'eval_steps_per_second': 18.077, 'epoch': 5.0}




20

In [29]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()


time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

Acesso negado - .
Arquivo n�o encontrado  - -NAME
Arquivo n�o encontrado  - -EXEC
Arquivo n�o encontrado  - RM
Arquivo n�o encontrado  - -RF
Arquivo n�o encontrado  - {}
Arquivo n�o encontrado  - +


In [30]:
results = {}
trainer_all = {}
s = splits[2]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: semantic


Map: 100%|██████████| 4180/4180 [00:00<00:00, 7929.89 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_13964\1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.164600,0.055713,"{'precision': 0.9480048367593712, 'recall': 0.9116279069767442, 'f1': 0.9294605809128632, 'number': 860}","{'precision': 0.8680555555555556, 'recall': 0.831946755407654, 'f1': 0.8496176720475787, 'number': 601}","{'precision': 0.9150865622423743, 'recall': 0.9495295124037639, 'f1': 0.9319899244332494, 'number': 1169}","{'precision': 0.9647058823529412, 'recall': 0.984984984984985, 'f1': 0.974739970282318, 'number': 666}",0.925364,0.925364,0.925364,0.986101,0.986101,0.931999,0.985956
2,0.018000,0.049820,"{'precision': 0.9565727699530516, 'recall': 0.9476744186046512, 'f1': 0.9521028037383177, 'number': 860}","{'precision': 0.9206896551724137, 'recall': 0.8885191347753744, 'f1': 0.9043183742591024, 'number': 601}","{'precision': 0.9501267962806424, 'recall': 0.9615055603079555, 'f1': 0.95578231292517, 'number': 1169}","{'precision': 0.9674556213017751, 'recall': 0.9819819819819819, 'f1': 0.9746646795827123, 'number': 666}",0.950167,0.948726,0.949446,0.989077,0.989077,0.941912,0.989050
3,0.008000,0.059391,"{'precision': 0.9457274826789839, 'recall': 0.9523255813953488, 'f1': 0.9490150637311704, 'number': 860}","{'precision': 0.8694942903752039, 'recall': 0.8868552412645591, 'f1': 0.8780889621087313, 'number': 601}","{'precision': 0.9460825610783488, 'recall': 0.960650128314799, 'f1': 0.9533106960950765, 'number': 1169}","{'precision': 0.9546783625730995, 'recall': 0.9804804804804805, 'f1': 0.9674074074074075, 'number': 666}",0.933731,0.949029,0.941318,0.988465,0.988465,0.941833,0.988413
4,0.002800,0.057300,"{'precision': 0.9545454545454546, 'recall': 0.9523255813953488, 'f1': 0.9534342258440047, 'number': 860}","{'precision': 0.9217687074829932, 'recall': 0.9018302828618968, 'f1': 0.9116904962153068, 'number': 601}","{'precision': 0.9461732548359967, 'recall': 0.962360992301112, 'f1': 0.9541984732824428, 'number': 1169}","{'precision': 0.9747023809523809, 'recall': 0.9834834834834835, 'f1': 0.9790732436472346, 'number': 666}",0.949803,0.952973,0.951386,0.990014,0.990014,0.948014,0.990010
5,0.001900,0.057903,"{'precision': 0.9450800915331807, 'recall': 0.9604651162790697, 'f1': 0.9527104959630911, 'number': 860}","{'precision': 0.8963815789473685, 'recall': 0.9068219633943427, 'f1': 0.9015715467328371, 'number': 601}","{'precision': 0.9580838323353293, 'recall': 0.9580838323353293, 'f1': 0.9580838323353293, 'number': 1169}","{'precision': 0.9761194029850746, 'recall': 0.9819819819819819, 'f1': 0.9790419161676646, 'number': 666}",0.947004,0.954187,0.950582,0.990137,0.990137,0.951732,0.990178


F1 Macro: 0.9567945361638434
F1 Micro: 0.9875929579602367
F1 Weighted: 0.9872881562957945
{'eval_loss': 0.0752422884106636, 'eval_LOC': {'precision': 0.97524467472654, 'recall': 0.9848837209302326, 'f1': 0.9800404975412207, 'number': 1720}, 'eval_MISC': {'precision': 0.8019662921348315, 'recall': 0.7897648686030428, 'f1': 0.7958188153310105, 'number': 723}, 'eval_ORG': {'precision': 0.983094262295082, 'recall': 0.9751016260162602, 'f1': 0.9790816326530613, 'number': 1968}, 'eval_PER': {'precision': 0.9662576687116564, 'recall': 0.9813084112149533, 'f1': 0.9737248840803708, 'number': 321}, 'eval_overall_precision': 0.9517664480643114, 'eval_overall_recall': 0.9507607776838546, 'eval_overall_f1': 0.9512633470768581, 'eval_overall_accuracy': 0.9875929579602367, 'eval_f1_micro': 0.9875929579602367, 'eval_f1_macro': 0.9567945361638434, 'eval_f1_weighted': 0.9872881562957945, 'eval_runtime': 8.5089, 'eval_samples_per_second': 491.249, 'eval_steps_per_second': 30.791, 'epoch': 5.0}




20

In [31]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

Acesso negado - .
Arquivo n�o encontrado  - -NAME
Arquivo n�o encontrado  - -EXEC
Arquivo n�o encontrado  - RM
Arquivo n�o encontrado  - -RF
Arquivo n�o encontrado  - {}
Arquivo n�o encontrado  - +


In [32]:
results = {}
trainer_all = {}
s = splits[3]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_len


Map: 100%|██████████| 4123/4123 [00:00<00:00, 12683.24 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_13964\1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.168900,0.023071,"{'precision': 0.9758342922899885, 'recall': 0.9691428571428572, 'f1': 0.9724770642201834, 'number': 875}","{'precision': 0.9211045364891519, 'recall': 0.8878326996197718, 'f1': 0.904162633107454, 'number': 526}","{'precision': 0.9553327256153145, 'recall': 0.9730733519034355, 'f1': 0.9641214351425943, 'number': 1077}","{'precision': 0.9838249286393911, 'recall': 0.9904214559386973, 'f1': 0.9871121718377088, 'number': 1044}",0.963961,0.964509,0.964235,0.993947,0.993947,0.965457,0.993932
2,0.017600,0.020384,"{'precision': 0.9826789838337182, 'recall': 0.9725714285714285, 'f1': 0.9775990809879379, 'number': 875}","{'precision': 0.9377431906614786, 'recall': 0.9163498098859315, 'f1': 0.9269230769230768, 'number': 526}","{'precision': 0.953026196928636, 'recall': 0.9795728876508821, 'f1': 0.9661172161172161, 'number': 1077}","{'precision': 0.989433237271854, 'recall': 0.9865900383141762, 'f1': 0.9880095923261392, 'number': 1044}",0.968821,0.970471,0.969645,0.994888,0.994888,0.971525,0.994885
3,0.007200,0.024113,"{'precision': 0.9715909090909091, 'recall': 0.9771428571428571, 'f1': 0.9743589743589745, 'number': 875}","{'precision': 0.9281663516068053, 'recall': 0.9334600760456274, 'f1': 0.9308056872037914, 'number': 526}","{'precision': 0.961218836565097, 'recall': 0.9665738161559888, 'f1': 0.9638888888888889, 'number': 1077}","{'precision': 0.9829222011385199, 'recall': 0.9923371647509579, 'f1': 0.9876072449952337, 'number': 1044}",0.965313,0.971891,0.968591,0.994384,0.994384,0.966361,0.994386
4,0.003200,0.025235,"{'precision': 0.9748858447488584, 'recall': 0.976, 'f1': 0.9754426042261565, 'number': 875}","{'precision': 0.9334600760456274, 'recall': 0.9334600760456274, 'f1': 0.9334600760456274, 'number': 526}","{'precision': 0.9526842584167425, 'recall': 0.9721448467966574, 'f1': 0.9623161764705882, 'number': 1077}","{'precision': 0.9857006673021925, 'recall': 0.9904214559386973, 'f1': 0.9880554228380316, 'number': 1044}",0.965070,0.972743,0.968891,0.994720,0.994720,0.970384,0.994730
5,0.001500,0.024440,"{'precision': 0.975, 'recall': 0.9805714285714285, 'f1': 0.9777777777777777, 'number': 875}","{'precision': 0.9342105263157895, 'recall': 0.9448669201520913, 'f1': 0.9395085066162571, 'number': 526}","{'precision': 0.9710820895522388, 'recall': 0.9665738161559888, 'f1': 0.968822708236389, 'number': 1077}","{'precision': 0.9866793529971456, 'recall': 0.9932950191570882, 'f1': 0.9899761336515513, 'number': 1044}",0.971146,0.974730,0.972935,0.995124,0.995124,0.970244,0.995131


F1 Macro: 0.9628923619789931
F1 Micro: 0.9929964568363191
F1 Weighted: 0.9929650907283436
{'eval_loss': 0.03675638511776924, 'eval_LOC': {'precision': 0.9701101206082853, 'recall': 0.9793541556379036, 'f1': 0.9747102212855637, 'number': 1889}, 'eval_MISC': {'precision': 0.8977777777777778, 'recall': 0.9115523465703971, 'f1': 0.9046126287505597, 'number': 1108}, 'eval_ORG': {'precision': 0.9627949183303085, 'recall': 0.9507168458781362, 'f1': 0.9567177637511272, 'number': 2232}, 'eval_PER': {'precision': 0.987012987012987, 'recall': 0.9899799599198397, 'f1': 0.9884942471235618, 'number': 1996}, 'eval_overall_precision': 0.9613152804642167, 'eval_overall_recall': 0.9630449826989619, 'eval_overall_f1': 0.9621793542142018, 'eval_overall_accuracy': 0.9929964568363191, 'eval_f1_micro': 0.9929964568363191, 'eval_f1_macro': 0.9628923619789931, 'eval_f1_weighted': 0.9929650907283436, 'eval_runtime': 11.1661, 'eval_samples_per_second': 369.244, 'eval_steps_per_second': 23.106, 'epoch': 5.0}




20

In [33]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()


time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

Acesso negado - .
Arquivo n�o encontrado  - -NAME
Arquivo n�o encontrado  - -EXEC
Arquivo n�o encontrado  - RM
Arquivo n�o encontrado  - -RF
Arquivo n�o encontrado  - {}
Arquivo n�o encontrado  - +


In [34]:
results = {}
trainer_all = {}
s = splits[4]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_rare


Map: 100%|██████████| 4123/4123 [00:00<00:00, 12064.47 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_13964\1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.168400,0.033743,"{'precision': 0.9647577092511013, 'recall': 0.9329073482428115, 'f1': 0.94856524093124, 'number': 939}","{'precision': 0.9027777777777778, 'recall': 0.8992094861660079, 'f1': 0.900990099009901, 'number': 506}","{'precision': 0.9328621908127208, 'recall': 0.9635036496350365, 'f1': 0.9479353680430879, 'number': 1096}","{'precision': 0.9808988764044944, 'recall': 0.9842164599774521, 'f1': 0.9825548677546426, 'number': 887}",0.949330,0.950992,0.950160,0.991126,0.991126,0.954025,0.991106
2,0.015100,0.030062,"{'precision': 0.9631966351209253, 'recall': 0.9755058572949947, 'f1': 0.9693121693121692, 'number': 939}","{'precision': 0.8918406072106262, 'recall': 0.9288537549407114, 'f1': 0.9099709583736689, 'number': 506}","{'precision': 0.9624197983501375, 'recall': 0.958029197080292, 'f1': 0.9602194787379972, 'number': 1096}","{'precision': 0.9920273348519362, 'recall': 0.9819616685456595, 'f1': 0.9869688385269122, 'number': 887}",0.959385,0.964702,0.962036,0.992792,0.992792,0.960568,0.992827
3,0.006700,0.030762,"{'precision': 0.9752421959095802, 'recall': 0.9648562300319489, 'f1': 0.9700214132762313, 'number': 939}","{'precision': 0.9329388560157791, 'recall': 0.9347826086956522, 'f1': 0.9338598223099704, 'number': 506}","{'precision': 0.9568733153638814, 'recall': 0.9717153284671532, 'f1': 0.9642372114078769, 'number': 1096}","{'precision': 0.9897959183673469, 'recall': 0.9842164599774521, 'f1': 0.9869983041266251, 'number': 887}",0.966774,0.967620,0.967196,0.993951,0.993951,0.968295,0.993947
4,0.003000,0.032695,"{'precision': 0.972310969116081, 'recall': 0.972310969116081, 'f1': 0.972310969116081, 'number': 939}","{'precision': 0.9421157684630739, 'recall': 0.932806324110672, 'f1': 0.9374379344587885, 'number': 506}","{'precision': 0.968094804010939, 'recall': 0.968978102189781, 'f1': 0.9685362517099864, 'number': 1096}","{'precision': 0.9898074745186863, 'recall': 0.9853438556933484, 'f1': 0.9875706214689266, 'number': 887}",0.971053,0.968786,0.969918,0.994241,0.994241,0.968072,0.994235
5,0.001100,0.033896,"{'precision': 0.971307120085016, 'recall': 0.9733759318423855, 'f1': 0.9723404255319148, 'number': 939}","{'precision': 0.948, 'recall': 0.9367588932806324, 'f1': 0.9423459244532805, 'number': 506}","{'precision': 0.9688644688644689, 'recall': 0.9653284671532847, 'f1': 0.96709323583181, 'number': 1096}","{'precision': 0.9875846501128668, 'recall': 0.9864712514092446, 'f1': 0.9870276367738295, 'number': 887}",0.971337,0.968786,0.970060,0.994168,0.994168,0.968055,0.994158


F1 Macro: 0.9472967463710434
F1 Micro: 0.9908379696154649
F1 Weighted: 0.9907914889533173
{'eval_loss': 0.052520234137773514, 'eval_LOC': {'precision': 0.958910227780259, 'recall': 0.9610564010743062, 'f1': 0.9599821149116924, 'number': 2234}, 'eval_MISC': {'precision': 0.8960055096418733, 'recall': 0.8880546075085324, 'f1': 0.892012341446692, 'number': 1465}, 'eval_ORG': {'precision': 0.9190361445783133, 'recall': 0.9185934489402697, 'f1': 0.9188147434353169, 'number': 2076}, 'eval_PER': {'precision': 0.9778393351800554, 'recall': 0.9832869080779945, 'f1': 0.9805555555555555, 'number': 2872}, 'eval_overall_precision': 0.9451120868962329, 'eval_overall_recall': 0.9458771828379785, 'eval_overall_f1': 0.9454944800878562, 'eval_overall_accuracy': 0.9908379696154649, 'eval_f1_micro': 0.9908379696154649, 'eval_f1_macro': 0.9472967463710434, 'eval_f1_weighted': 0.9907914889533173, 'eval_runtime': 16.0609, 'eval_samples_per_second': 256.71, 'eval_steps_per_second': 16.064, 'epoch': 5.0}




20

In [35]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

Acesso negado - .
Arquivo n�o encontrado  - -NAME
Arquivo n�o encontrado  - -EXEC
Arquivo n�o encontrado  - RM
Arquivo n�o encontrado  - -RF
Arquivo n�o encontrado  - {}
Arquivo n�o encontrado  - +


In [36]:
results = {}
trainer_all = {}
s = splits[5]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: advs
Selecionando 4123 sentenças para teste…
  0 selecionadas…
  21 selecionadas…
  41 selecionadas…
  65 selecionadas…
  84 selecionadas…
  107 selecionadas…
  127 selecionadas…
  152 selecionadas…
  174 selecionadas…
  194 selecionadas…
  215 selecionadas…
  234 selecionadas…
  258 selecionadas…
  283 selecionadas…
  306 selecionadas…
  325 selecionadas…
  343 selecionadas…
  368 selecionadas…
  390 selecionadas…
  414 selecionadas…
  435 selecionadas…
  460 selecionadas…
  480 selecionadas…
  501 selecionadas…
  522 selecionadas…
  544 selecionadas…
  561 selecionadas…
  573 selecionadas…
  587 selecionadas…
  605 selecionadas…
  624 selecionadas…
  645 selecionadas…
  670 selecionadas…
  689 selecionadas…
  705 selecionadas…
  730 selecionadas…
  744 selecionadas…
  768 selecionadas…
  789 selecionadas…
  808 selecionadas…
  828 selecionadas…
  849 selecionadas…
  870 selecionadas…
  883 selecionadas…
  904 selecionadas…
  922 selecionadas…
  938 selecionadas…


Map: 100%|██████████| 4123/4123 [00:00<00:00, 16475.41 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_13964\1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.172800,0.027732,"{'precision': 0.9563492063492064, 'recall': 0.9707955689828801, 'f1': 0.9635182408795602, 'number': 993}","{'precision': 0.8695652173913043, 'recall': 0.9025270758122743, 'f1': 0.8857395925597874, 'number': 554}","{'precision': 0.9427710843373494, 'recall': 0.9418254764292878, 'f1': 0.9422980431510285, 'number': 997}","{'precision': 0.97997997997998, 'recall': 0.9888888888888889, 'f1': 0.9844142785319256, 'number': 990}",0.945221,0.956989,0.951069,0.992151,0.992151,0.955956,0.992161
2,0.019400,0.032828,"{'precision': 0.9571713147410359, 'recall': 0.9677744209466264, 'f1': 0.9624436654982474, 'number': 993}","{'precision': 0.8783068783068783, 'recall': 0.8989169675090253, 'f1': 0.8884924174843889, 'number': 554}","{'precision': 0.9239024390243903, 'recall': 0.9498495486459378, 'f1': 0.9366963402571712, 'number': 997}","{'precision': 0.9731610337972167, 'recall': 0.9888888888888889, 'f1': 0.9809619238476954, 'number': 990}",0.939756,0.957838,0.948711,0.991768,0.991768,0.953975,0.991768
3,0.007200,0.031470,"{'precision': 0.9638916750250752, 'recall': 0.9677744209466264, 'f1': 0.9658291457286432, 'number': 993}","{'precision': 0.8898601398601399, 'recall': 0.9187725631768953, 'f1': 0.9040852575488455, 'number': 554}","{'precision': 0.9606854838709677, 'recall': 0.9558676028084253, 'f1': 0.9582704876822525, 'number': 997}","{'precision': 0.9879154078549849, 'recall': 0.990909090909091, 'f1': 0.989409984871407, 'number': 990}",0.957794,0.963214,0.960497,0.993619,0.993619,0.962596,0.993609
4,0.003400,0.031185,"{'precision': 0.9696048632218845, 'recall': 0.9637462235649547, 'f1': 0.9666666666666667, 'number': 993}","{'precision': 0.9076376554174067, 'recall': 0.9223826714801444, 'f1': 0.9149507609668756, 'number': 554}","{'precision': 0.9551345962113659, 'recall': 0.9608826479438315, 'f1': 0.958, 'number': 997}","{'precision': 0.9829659318637275, 'recall': 0.990909090909091, 'f1': 0.9869215291750504, 'number': 990}",0.959448,0.964063,0.961750,0.993779,0.993779,0.963657,0.993767
5,0.001100,0.031885,"{'precision': 0.9696663296258847, 'recall': 0.9657603222557906, 'f1': 0.9677093844601412, 'number': 993}","{'precision': 0.9063604240282686, 'recall': 0.9259927797833934, 'f1': 0.9160714285714284, 'number': 554}","{'precision': 0.9628886659979939, 'recall': 0.9628886659979939, 'f1': 0.9628886659979939, 'number': 997}","{'precision': 0.9839357429718876, 'recall': 0.98989898989899, 'f1': 0.986908358509567, 'number': 990}",0.961669,0.965478,0.963570,0.994193,0.994193,0.966101,0.994181


F1 Macro: 0.9437451951098361
F1 Micro: 0.9905509105486199
F1 Weighted: 0.9905183736201186
{'eval_loss': 0.058429472148418427, 'eval_LOC': {'precision': 0.9609423434593924, 'recall': 0.9573810994441013, 'f1': 0.9591584158415841, 'number': 1619}, 'eval_MISC': {'precision': 0.870561282932417, 'recall': 0.880648899188876, 'f1': 0.8755760368663594, 'number': 863}, 'eval_ORG': {'precision': 0.9555832295558323, 'recall': 0.9516329061595701, 'f1': 0.9536039768019884, 'number': 2419}, 'eval_PER': {'precision': 0.973435655253837, 'recall': 0.981547619047619, 'f1': 0.9774748073503259, 'number': 1680}, 'eval_overall_precision': 0.9502200637426013, 'eval_overall_recall': 0.9513751709466647, 'eval_overall_f1': 0.9507972665148063, 'eval_overall_accuracy': 0.9905509105486199, 'eval_f1_micro': 0.9905509105486199, 'eval_f1_macro': 0.9437451951098361, 'eval_f1_weighted': 0.9905183736201186, 'eval_runtime': 10.3224, 'eval_samples_per_second': 399.421, 'eval_steps_per_second': 24.994, 'epoch': 5.0}




20